# 第 06 课｜复利与贴现

> **数据身份**：固定随机种子生成的模拟教学数据，不是真实证券行情。  
> **环境**：`pkm-quant` · Python 3.11 · 默认离线运行。  
> **课程原始材料**：`06_youtube-3YpcWCIDGXo/index.html`  
> **所属模块**：P1-M2 · 收益、时间价值与风险

## 本节量化问题

如何把不同时点的现金流换算到同一个时点比较？

## 准确定义与策略报告位置

复利把当前价值推到未来，贴现把未来现金流拉回当前；NPV 使用外部要求回报率，IRR 是使 NPV 为零的内含收益率。

- **核心关系**：FV=PV(1+r)^N；NPV(r)=Σ CF_t/(1+r)^t；IRR 是使 NPV=0 的折现率。
- **写入报告**：现金流时点、机会成本和资本占用
- **P2 衔接**：P2 第 32–33 课：数组计算和数值求解。


## 运行环境与数据契约

先确认项目、数据目录和数据身份。后续联网数据源必须通过适配层进入，不能改变本 Notebook 的研究计算口径。

In [1]:
from pathlib import Path
import json
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / "data" / "fixtures").exists():
    ROOT = ROOT.parent
DATA_DIR = ROOT / "data" / "fixtures"
OUTPUT_DIR = ROOT / "outputs"
OUTPUT_DIR.mkdir(exist_ok=True)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

RANDOM_SEED = 365
print("项目:", ROOT)
print("数据:", DATA_DIR)
print("数据身份: 固定随机种子生成的模拟教学数据，不是真实证券行情")


项目: /Users/mojiawen/Documents/学习资料归档/_暂存/personal_knowledge_manager/量化交易/实验/p1-quant-foundations
数据: /Users/mojiawen/Documents/学习资料归档/_暂存/personal_knowledge_manager/量化交易/实验/p1-quant-foundations/data/fixtures
数据身份: 固定随机种子生成的模拟教学数据，不是真实证券行情


## 复利与贴现是一对逆运算

同一利率和期限下，先复利再贴现应回到原始本金。

In [2]:
present_value = 10_000.0
annual_rate = 0.08
years = 10

future_value = present_value * (1.0 + annual_rate) ** years
discounted_back = future_value / (1.0 + annual_rate) ** years

print(f"10 年后价值: {future_value:,.2f}")
print(f"贴现回今天: {discounted_back:,.2f}")
assert np.isclose(discounted_back, present_value)

10 年后价值: 21,589.25
贴现回今天: 10,000.00


## NPV 使用外部折现率，IRR 由现金流反求

现金流第 0 期为投入。这里用 `brentq` 在给定区间寻找使 NPV 为零的根；区间和现金流符号变化都会影响是否有解。

In [3]:
from scipy.optimize import brentq

cashflows = np.array([-1_000.0, 380.0, 420.0, 460.0])

def npv(rate, flows):
    periods = np.arange(len(flows))
    return np.sum(flows / (1.0 + rate) ** periods)

required_return = 0.06
project_npv = npv(required_return, cashflows)
project_irr = brentq(lambda rate: npv(rate, cashflows), -0.99, 2.0)

print(f"NPV @ 6%: {project_npv:.2f}")
print(f"IRR: {project_irr:.2%}")
assert abs(npv(project_irr, cashflows)) < 1e-8

NPV @ 6%: 118.51
IRR: 12.09%


## 方法边界与阶段验收

IRR 可能多解且隐含再投资假设；不规则现金流需要按实际日期计算 XIRR。

完成本课后应能：

1. 不看课程材料准确解释核心概念和公式符号；
2. 用小样本手工核对 Notebook 的关键结果；
3. 说明结果能支持什么判断、不能支持什么判断；
4. 指出它在策略报告中的位置，以及仍缺少的数据、成本或稳健性证据。
